# House Price Prediction — Data Cleaning, Modeling & Export
Amr Abdelfatah Mahmoud Abdelmonem


## 2.1 Load & Inspect

In [ ]:
# Amr Abdelfatah Mahmoud Abdelmonem
import pandas as pd

df = pd.read_csv("data/house_prices.csv")
df.shape
df.head()
df.info()
df.describe()
df.isna().mean().sort_values(ascending=False)


**Rows/columns:** the dataset has thousands of listings with a mix of numeric (Bathroom, Balcony, Car Parking) and text columns that look numeric but need parsing (Amount(in rupees), Carpet Area, Super Area, Floor). location and Society are high-cardinality text columns, and several columns (Society, facing, overlooking, Plot Area) have the most missing values.

## 2.2 Exploratory Data Analysis (EDA)

In [ ]:
# Amr Abdelfatah Mahmoud Abdelmonem
import matplotlib.pyplot as plt
import seaborn as sns

def parse_amount(x):
    if not isinstance(x, str):
        return None
    x = x.strip().lower()
    try:
        if "lac" in x:
            return float(x.replace("lac", "").strip()) * 1e5
        if "cr" in x:
            return float(x.replace("cr", "").strip()) * 1e7
        return float(x.replace(",", ""))
    except ValueError:
        return None

df["price_clean"] = df["Amount(in rupees)"].apply(parse_amount)
df = df.dropna(subset=["price_clean"])

sns.histplot(df["price_clean"], log_scale=True)
plt.title("Price distribution (log scale)")
plt.show()


Price is heavily right-skewed, which is why a log scale is used and why predicting log(price) is worth trying later.

In [ ]:
# Amr Abdelfatah Mahmoud Abdelmonem
def parse_area(x):
    if not isinstance(x, str):
        return None
    x = x.strip().lower()
    try:
        if "sqm" in x:
            return float(x.replace("sqm", "").strip()) * 10.764
        if "sqft" in x:
            return float(x.replace("sqft", "").strip())
        return float(x)
    except ValueError:
        return None

df["carpet_area_sqft"] = df["Carpet Area"].apply(parse_area)

plt.figure()
plt.scatter(df["carpet_area_sqft"], df["price_clean"], alpha=0.2)
plt.xlabel("Carpet area (sqft)")
plt.ylabel("Price")
plt.title("Price vs carpet area")
plt.show()


Price generally rises with carpet area, with a lot of spread coming from location and finish quality.

In [ ]:
# Amr Abdelfatah Mahmoud Abdelmonem
top15 = df.groupby("location")["price_clean"].mean().sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 5))
top15.plot(kind="bar")
plt.title("Average price by top-15 locations")
plt.ylabel("Average price")
plt.xticks(rotation=75)
plt.tight_layout()
plt.show()


Average price varies a lot by location, confirming location is an important feature.

In [ ]:
# Amr Abdelfatah Mahmoud Abdelmonem
df["bathroom"] = pd.to_numeric(df["Bathroom"], errors="coerce")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.boxplot(x="Furnishing", y="price_clean", data=df, ax=axes[0])
axes[0].set_title("Price by furnishing status")
sns.boxplot(x="bathroom", y="price_clean", data=df, ax=axes[1])
axes[1].set_title("Price by number of bathrooms")
plt.tight_layout()
plt.show()


Furnished listings and listings with more bathrooms tend to have higher median prices.

## 2.3 Cleaning & Feature Engineering

In [ ]:
# Amr Abdelfatah Mahmoud Abdelmonem
def parse_floor(x):
    if not isinstance(x, str):
        return None
    x = x.strip().lower()
    first = x.split("out of")[0].strip()
    if first == "ground":
        return 0
    if first == "basement":
        return -1
    try:
        return float(first)
    except ValueError:
        return None

df["super_area_sqft"] = df["Super Area"].apply(parse_area)
df["carpet_area_sqft"] = df["carpet_area_sqft"].fillna(df["super_area_sqft"])
df = df.dropna(subset=["carpet_area_sqft"])

df["floor_num"] = df["Floor"].apply(parse_floor)

df["bathroom"] = df["bathroom"].fillna(df["bathroom"].median())

df["balcony"] = pd.to_numeric(df["Balcony"], errors="coerce")
df["balcony"] = df["balcony"].fillna(0)

df["car_parking"] = pd.to_numeric(df["Car Parking"], errors="coerce")
df["car_parking"] = df["car_parking"].fillna(0)

TOP_N_LOCATIONS = 50
top_locations = df["location"].value_counts().head(TOP_N_LOCATIONS).index
df["location_grouped"] = df["location"].where(df["location"].isin(top_locations), "other")

df["Furnishing"] = df["Furnishing"].fillna("Unfurnished")
df["Transaction"] = df["Transaction"].fillna("Resale")
df["Ownership"] = df["Ownership"].fillna("Freehold")
df["facing"] = df["facing"].fillna("Not Available")

df = df.drop(columns=["Index", "Title", "Description", "Dimensions"], errors="ignore")

df["price_per_sqft"] = df["price_clean"] / df["carpet_area_sqft"]
low, high = df["price_per_sqft"].quantile([0.01, 0.99])
df = df[(df["price_per_sqft"] >= low) & (df["price_per_sqft"] <= high)]

df.shape


## 2.4 Build a Pipeline & Train

In [ ]:
# Amr Abdelfatah Mahmoud Abdelmonem
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression

numeric_features = ["carpet_area_sqft", "floor_num", "bathroom", "balcony"]
categorical_features = ["location_grouped", "Furnishing", "Transaction", "Ownership", "facing"]

preprocessor = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                       ("scale", StandardScaler())]), numeric_features),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                       ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical_features),
])

X = df[numeric_features + categorical_features]
y = df["price_clean"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    "LinearRegression": LinearRegression(),
    "RandomForestRegressor": RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    "GradientBoostingRegressor": GradientBoostingRegressor(random_state=42),
}

fitted = {}
for name, reg in models.items():
    pipe = Pipeline([("prep", preprocessor), ("reg", reg)])
    pipe.fit(X_train, y_train)
    fitted[name] = pipe


## 2.5 Evaluate

In [ ]:
# Amr Abdelfatah Mahmoud Abdelmonem
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

results = {}
for name, pipe in fitted.items():
    pred = pipe.predict(X_test)
    results[name] = {
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": root_mean_squared_error(y_test, pred),
        "R2": r2_score(y_test, pred),
    }

pd.DataFrame(results).T


In [ ]:
# Amr Abdelfatah Mahmoud Abdelmonem
best_name = max(results, key=lambda k: results[k]["R2"])
best_model = fitted[best_name]

pred = best_model.predict(X_test)
plt.figure()
plt.scatter(y_test, pred, alpha=0.2)
plt.xlabel("Actual price")
plt.ylabel("Predicted price")
plt.title(f"Predicted vs actual ({best_name})")
plt.show()

best_name, results[best_name]


**Model choice:** the model with the highest R² and lowest RMSE/MAE on the held-out test set is selected as the final model, since those metrics best reflect how well it generalises to unseen listings rather than how well it memorised the training data.

## 2.6 Export the Model

In [ ]:
# Amr Abdelfatah Mahmoud Abdelmonem
import joblib
import json

joblib.dump(best_model, "house_price.pkl")

loaded = joblib.load("house_price.pkl")
sample = X_test.iloc[[0]]
print("Reloaded prediction:", loaded.predict(sample))

json.dump(sorted(df["location_grouped"].unique().tolist()), open("locations.json", "w"))


**Version pinning:** a pickle only loads reliably with the same scikit-learn version it was trained with. Note your version below and pin it in the backend `requirements.txt`.

In [ ]:
# Amr Abdelfatah Mahmoud Abdelmonem
import sklearn
sklearn.__version__
